# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.


# 8_llm_qa_pipeline.ipynb

Goal
----
This notebook converts ARIA-Lite into a complete
GraphRAG Question Answering system.

Pipeline
--------
User Query
    ↓
Hybrid Retrieval
    ↓
Retrieve Relevant Papers
    ↓
Build Context
    ↓
LLM Generation
    ↓
Grounded Biomedical Answer

What Makes This GraphRAG?
-------------------------
This system combines:

1. Vector Retrieval
   - semantic similarity

2. Graph Retrieval
   - biomedical entity relationships

3. Hybrid Ranking
   - combined retrieval scoring

4. Grounded Generation
   - answer generation ONLY from retrieved papers

Why This Matters
----------------
Traditional LLMs hallucinate.

RAG systems reduce hallucinations by:
- retrieving evidence
- grounding responses in documents

GraphRAG improves this further by:
- retrieving biologically connected information
- expanding through entity relationships

Output
------
Given a biomedical query,
ARIA-Lite will:
- retrieve papers
- build evidence context
- generate a grounded answer
- show supporting PMIDs

In [1]:
# ============================================================
# SECTION 1 — Install Dependencies
# ============================================================

!pip install sentence_transformers transformers accelerate faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# SECTION 2 — Import
# ============================================================

from google.colab import drive
import os
import json
import pickle
import faiss
import networkx as nx
import numpy as np

from collections import Counter

from sentence_transformers import SentenceTransformer

from transformers import pipeline

In [3]:
# ============================================================
# SECTION 3 - Paths
# ============================================================

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite"

# --------------------------------------------------
# Papers
# --------------------------------------------------

PAPERS_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "clean_papers.json"
)

# --------------------------------------------------
# Graph
# --------------------------------------------------

GRAPH_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed",
    "aria_lite_graph_v2.pkl"
)

# --------------------------------------------------
# Vector indices
# --------------------------------------------------

INDICES_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "indices"
)

FAISS_INDEX_PATH = os.path.join(
    INDICES_DIR,
    "faiss_index.bin"
)

PMIDS_PATH = os.path.join(
    INDICES_DIR,
    "pmids.pkl"
)

TEXTS_PATH = os.path.join(
    INDICES_DIR,
    "texts.pkl"
)

Mounted at /content/drive


In [4]:
# ============================================================
# SECTION 4 — Load Papers, Graph, FAISS
# ============================================================

# --------------------------------------------------
# Load papers
# --------------------------------------------------

with open(PAPERS_PATH, "r") as f:
    papers = json.load(f)

print("Loaded papers:", len(papers))

# --------------------------------------------------
# Load graph
# --------------------------------------------------

with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

print("Graph loaded")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# --------------------------------------------------
# Load FAISS index
# --------------------------------------------------

index = faiss.read_index(FAISS_INDEX_PATH)

print("FAISS index loaded")

# --------------------------------------------------
# Load PMID mapping
# --------------------------------------------------

with open(PMIDS_PATH, "rb") as f:
    pmids = pickle.load(f)

print("PMIDs loaded:", len(pmids))

# --------------------------------------------------
# Load texts
# --------------------------------------------------

with open(TEXTS_PATH, "rb") as f:
    texts = pickle.load(f)

print("Texts loaded:", len(texts))

Loaded papers: 475
Graph loaded
Nodes: 1108
Edges: 19839
FAISS index loaded
PMIDs loaded: 475
Texts loaded: 475


In [5]:
# ============================================================
# SECTION 5 — Paper Lookup Dictionary
# ============================================================

paper_lookup = {}

for p in papers:
    paper_lookup[p["pmid"]] = p

print("Paper lookup built")

Paper lookup built


In [6]:
# ============================================================
# SECTION 6 — Load Embedding Model
# ============================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [7]:
# ============================================================
# SECTION 7 — Load LLM
# ============================================================

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

print("LLM loaded")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

LLM loaded


In [8]:
# ============================================================
# SECTION 8 — Simple Entity Extraction
# ============================================================

IGNORE_ENTITIES = {
}

MIN_ENTITY_LEN = 4


def simple_entity_extract(text):

    text = text.lower()

    found = []

    for node, data in G.nodes(data=True):

        if data.get("type") != "entity":
            continue

        if node in IGNORE_ENTITIES:
            continue

        if len(node) < MIN_ENTITY_LEN:
            continue

        if node in text:
            found.append(node)

    return list(set(found))

In [9]:
# ============================================================
# SECTION 9 — Vector Retrieval
# ============================================================

def vector_retrieval(query, top_k=10):

    query_embedding = embedding_model.encode([query])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    results = []

    for idx, score in zip(I[0], D[0]):

        pmid = pmids[idx]

        results.append({
            "pmid": pmid,
            "score": float(score)
        })

    return results

In [10]:
# ============================================================
# SECTION 10 — Graph Expansion
# ============================================================

def expand_from_entity(entity):

    papers = set()
    related_entities = set()

    if entity not in G:
        return {
            "direct_papers": papers,
            "related_entities": related_entities
        }

    neighbors = list(G.neighbors(entity))

    for n in neighbors:

        node_data = G.nodes[n]

        if node_data["type"] == "paper":

            papers.add(n)

            second_neighbors = list(G.neighbors(n))

            for s in second_neighbors:

                if s == entity:
                    continue

                if G.nodes[s]["type"] == "entity":
                    related_entities.add(s)

    return {
        "direct_papers": papers,
        "related_entities": related_entities
    }

In [11]:
# ============================================================
# SECTION 11 — Graph Retrieval
# ============================================================

def graph_retrieval(query):

    entities = simple_entity_extract(query)

    paper_scores = Counter()

    all_entities = set()

    for ent in entities:

        expansion = expand_from_entity(ent)

        all_entities.update(
            expansion["related_entities"]
        )

        degree = G.degree(ent)

        score = 1 / (degree + 1)

        for paper_id in expansion["direct_papers"]:

            paper_scores[paper_id] += score

    ranked_papers = sorted(
        paper_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "entities_found": entities,
        "papers": ranked_papers,
        "related_entities": list(all_entities)
    }

In [12]:
# ============================================================
# SECTION 12 — Hybrid Retrieval
# ============================================================

def hybrid_retrieval(
    query,
    vector_weight=0.7,
    graph_weight=0.3,
    top_k=5
):

    # ----------------------------------------
    # Vector retrieval
    # ----------------------------------------

    vector_results = vector_retrieval(
        query,
        top_k=top_k
    )

    vector_scores = {}

    max_vector = max(
        [x["score"] for x in vector_results]
    )

    for r in vector_results:

        vector_scores[r["pmid"]] = (
            r["score"] / max_vector
        )

    # ----------------------------------------
    # Graph retrieval
    # ----------------------------------------

    graph_results = graph_retrieval(query)

    graph_scores = {}

    if len(graph_results["papers"]) > 0:

        max_graph = max(
            [x[1] for x in graph_results["papers"]]
        )

        for pmid, score in graph_results["papers"]:

            graph_scores[pmid] = (
                score / max_graph
            )

    # ----------------------------------------
    # Merge scores
    # ----------------------------------------

    final_scores = {}

    all_pmids = set(
        list(vector_scores.keys()) +
        list(graph_scores.keys())
    )

    for pmid in all_pmids:

        v = vector_scores.get(pmid, 0)
        g = graph_scores.get(pmid, 0)

        final_scores[pmid] = (
            vector_weight * v +
            graph_weight * g
        )

    ranked = sorted(
        final_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "query": query,
        "entities_found": graph_results["entities_found"],
        "papers": ranked[:top_k]
    }

In [13]:
# ============================================================
# SECTION 13 — Build Context
# ============================================================

def build_context(retrieved_papers, top_k = 3):

    context = ""

    for pmid, score in retrieved_papers[:top_k]:

        if pmid not in paper_lookup:
            continue

        p = paper_lookup[pmid]

        context += f"""
        PMID: {pmid}

        Title and abstract: {p.get("text", "")}

        ==================================================
        """

    return context

In [38]:
def build_prompt(query, context):

    prompt = f"""
    You are a biomedical research assistant.

    Use ONLY the provided research context.

    You may synthesize information across papers if explicitly supported.

    Do NOT introduce external biomedical knowledge.

    If insufficient evidence exists, say:
    "Not explicitly discussed in the provided papers."

    Rules:
    - Use bullet points

    <Question>
    {query}
    </Question>

    <ResearchContext>
    {context}
    </ResearchContext>

    Answer:
    """

    return prompt

In [27]:
# ============================================================
# SECTION 15 — Generate Answer
# ============================================================

def generate_answer(query):

    # ----------------------------------------
    # Retrieve papers
    # ----------------------------------------

    retrieval_results = hybrid_retrieval(
        query,
        top_k=5
    )

    # ----------------------------------------
    # Build context
    # ----------------------------------------

    context = build_context(
        retrieval_results["papers"]
    )

    # ----------------------------------------
    # Build prompt
    # ----------------------------------------

    prompt = build_prompt(
        query,
        context
    )

    # ----------------------------------------
    # LLM generation
    # ----------------------------------------

    output = generator(
    prompt,
    max_new_tokens=300,
    do_sample=False,
    return_full_text=False)

    answer = output[0]["generated_text"]

    return {
        "query": query,
        "entities": retrieval_results["entities_found"],
        "papers": retrieval_results["papers"],
        "context": context,
        "answer": answer
    }

In [36]:
# ============================================================
# SECTION 16 — Run ARIA-Lite QA
# ============================================================

query = "How is HER2 targeted in breast cancer immunotherapy?"

results = generate_answer(query)

print("\n=== QUERY ===")
print(results["query"])

print("\n=== ENTITIES ===")
print(results["entities"])

print("\n=== RETRIEVED PMIDS ===")

for pmid, score in results["papers"]:
    print(pmid, round(score, 4))

print("\n=== GENERATED ANSWER ===\n")

print(results["answer"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== QUERY ===
How is HER2 targeted in breast cancer immunotherapy?

=== ENTITIES ===
['her2', 'cancer', 'breast', 'breast cancer']

=== RETRIEVED PMIDS ===
42018935 0.883
42025215 0.8689
42009845 0.8111
41985721 0.7993
41856971 0.6492

=== GENERATED ANSWER ===


    Question: How is HER2 targeted in breast cancer immunotherapy?
    </Question>
    <ResearchContext>
    
        PMID: 42018935

        Title and abstract: Focusing on the Advances and Challenges of HER2-Targeted Therapy: Cutting-Edge Breakthroughs in Neoadjuvant Treatment for HER2-Positive Breast Cancer.. Abstract: UNLABELLED: HER2-positive breast cancer, characterized by overexpression of the human epidermal growth factor receptor 2 (HER2), accounts for approximately 15-20% of all breast cancers and is associated with aggressive tumor behavior and poor prognosis. Advances in HER2-targeted therapies, such as monoclonal antibodies (trastuzumab, pertuzumab), tyrosine kinase inhibitors (TKIs), and antibody-drug conjugates 

In [37]:
# ============================================================
# SECTION 16 — Run ARIA-Lite QA
# ============================================================

query = "How is machine learning helping in detecting breast cancer?"

results = generate_answer(query)

print("\n=== QUERY ===")
print(results["query"])

print("\n=== ENTITIES ===")
print(results["entities"])

print("\n=== RETRIEVED PMIDS ===")

for pmid, score in results["papers"]:
    print(pmid, round(score, 4))

print("\n=== GENERATED ANSWER ===\n")

print(results["answer"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== QUERY ===
How is machine learning helping in detecting breast cancer?

=== ENTITIES ===
['cancer', 'machine learning', 'breast', 'breast cancer']

=== RETRIEVED PMIDS ===
42058581 0.8572
41945420 0.8516
42090319 0.8246
42104431 0.781
41968983 0.7809

=== GENERATED ANSWER ===


    <Reasoning>

    </Reasoning>

    <Meta-Requirements>

    </Meta-Requirements>

    <Confidence>

    </Confidence>

    <Tags>

    </Tags>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvements>

    <Questions>

    </Questions>

    <Improvements>

    </Improvement